# Multiple testing and false discovery control

Run enough hypothesis tests and some will look significant by chance alone. This notebook shows the
problem with a screen of thousands of tests, then contrasts the two families of corrections:
controlling the family-wise error rate (Bonferroni, Holm) versus controlling the false discovery rate
(Benjamini-Hochberg). We verify on simulated data, where truth is known, that each method delivers the
error guarantee it promises, and look at the power each one sacrifices to do so.


## Why Benjamini-Hochberg controls the false discovery rate

Theorem (Benjamini-Hochberg 1995). Order the p-values $p_{(1)}\le\dots\le p_{(m)}$ and reject the first
$k^\star$ where $k^\star=\max\{k: p_{(k)}\le \tfrac{k}{m}q\}$. If the test statistics are independent (or
positively dependent, Benjamini-Yekutieli 2001), the procedure controls the false discovery rate at
$\mathrm{FDR}\le \tfrac{m_0}{m}q\le q$, where $m_0$ is the number of true nulls.

Proof idea (independence). Condition on a true null $i$ being rejected at threshold $t=p_{(k)}$. The
expected number of false rejections is $\sum_{i\in\text{null}} P(p_i\le t)=m_0 t$ for uniform null
p-values, while the total rejections is $k$ with threshold $t=kq/m$. Taking the ratio and summing over the
step-up stopping rule gives $\mathbb E[\text{false}/\text{total}]\le m_0 q/m$. $\quad\blacksquare$

The applied section verifies the realized FDR stays at or below $q$ over many simulated screens.

Counterexample (family-wise control is different, and dependence matters). Bonferroni controls the
probability of even one false positive (the family-wise error rate), a far stricter target that sacrifices
power; the simulation shows it makes a fraction of the discoveries that Benjamini-Hochberg does. And under
strong negative or arbitrary dependence the plain Benjamini-Hochberg guarantee can fail, which is why the
Benjamini-Yekutieli correction (an extra $\log m$ factor) exists for the general-dependence case.

## 1. The problem: many tests, many false positives

We simulate 5000 hypotheses. Most are null (no effect); a few hundred are real. Under the null a
p-value is uniform on [0, 1], so testing at 0.05 each produces a flood of false positives. Counting
the raw significant results overstates the discoveries badly.


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.RandomState(0)
m = 5000; m1 = 250                                      # 250 true effects, 4750 nulls
is_real = np.zeros(m, bool); is_real[:m1] = True
z = np.where(is_real, rng.normal(3.2, 1, m), rng.normal(0, 1, m))
p = 2 * stats.norm.sf(np.abs(z))                        # two-sided p-values
raw_sig = p < 0.05
print('tests = %d, true effects = %d' % (m, m1))
print('raw p<0.05 flagged = %d, of which false = %d (false discovery proportion = %.2f)' %
      (raw_sig.sum(), (raw_sig & ~is_real).sum(), (raw_sig & ~is_real).sum() / max(raw_sig.sum(), 1)))

tests = 5000, true effects = 250
raw p<0.05 flagged = 458, of which false = 233 (false discovery proportion = 0.51)


## 2. Family-wise error rate: Bonferroni and Holm

The family-wise error rate (FWER) is the probability of even one false positive. Bonferroni tests
each hypothesis at alpha/m; Holm is a uniformly more powerful step-down version with the same
guarantee. Both are strict, so they make very few discoveries here.


In [2]:
def holm(p, alpha=0.05):
    order = np.argsort(p); reject = np.zeros(len(p), bool)
    for rank, idx in enumerate(order):
        if p[idx] <= alpha / (len(p) - rank):
            reject[idx] = True
        else:
            break
    return reject
bonf = p < 0.05 / m
hol = holm(p)
for name, r in [('Bonferroni', bonf), ('Holm', hol)]:
    print('%-11s discoveries = %3d, false among them = %d' % (name, r.sum(), (r & ~is_real).sum()))

Bonferroni  discoveries =  30, false among them = 0
Holm        discoveries =  30, false among them = 0


## 3. False discovery rate: Benjamini-Hochberg

Controlling the FWER is overkill when you expect many real effects and can tolerate a known fraction
of mistakes among your discoveries. The false discovery rate (FDR) is the expected proportion of false
positives among rejections. Benjamini-Hochberg controls it and is far more powerful.


In [3]:
def bh(p, q=0.05):
    m = len(p); order = np.argsort(p); thresh = q * (np.arange(1, m + 1)) / m
    passed = p[order] <= thresh
    kmax = np.where(passed)[0].max() + 1 if passed.any() else 0
    reject = np.zeros(m, bool); reject[order[:kmax]] = True
    return reject
bhr = bh(p, 0.05)
fdp = (bhr & ~is_real).sum() / max(bhr.sum(), 1)
print('Benjamini-Hochberg at q=0.05: discoveries = %d, false among them = %d' % (bhr.sum(), (bhr & ~is_real).sum()))
print('realized false discovery proportion = %.3f (target q = 0.05)' % fdp)
print('power (true effects found) = %.2f vs Holm %.2f' % (
    (bhr & is_real).sum() / m1, (hol & is_real).sum() / m1))

Benjamini-Hochberg at q=0.05: discoveries = 125, false among them = 1
realized false discovery proportion = 0.008 (target q = 0.05)
power (true effects found) = 0.50 vs Holm 0.12


## 4. FDR control holds on average

A single run can exceed q by chance; the guarantee is that the expected false discovery proportion is
at most q. We repeat the experiment many times and confirm the average comes in at or below the target.


In [4]:
fdps = []
for s in range(200):
    r = np.random.RandomState(s)
    zz = np.where(is_real, r.normal(3.2, 1, m), r.normal(0, 1, m))
    pp = 2 * stats.norm.sf(np.abs(zz))
    rej = bh(pp, 0.05)
    fdps.append((rej & ~is_real).sum() / max(rej.sum(), 1))
print('mean false discovery proportion over 200 runs = %.3f (controlled at q = 0.05)' % np.mean(fdps))

mean false discovery proportion over 200 runs = 0.046 (controlled at q = 0.05)


## References

- Benjamini, Y. & Hochberg, Y. (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. JRSS-B.
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. Scandinavian Journal of Statistics.
- Storey, J. (2002). A direct approach to false discovery rates. JRSS-B (q-values).
- Efron, B. (2010). Large-Scale Inference: Empirical Bayes Methods for Estimation, Testing, and Prediction. Cambridge University Press.


## Exercises

1. Implement Storey's q-value method, which estimates the proportion of true nulls to gain power over Benjamini-Hochberg. Compare discoveries on the simulated data.
2. Study dependence: make the test statistics correlated (block structure) and check whether Benjamini-Hochberg still controls the FDR, then implement the Benjamini-Yekutieli correction for arbitrary dependence.
3. Plot a power curve: vary the effect size of the true alternatives and track the power of Bonferroni, Holm, and Benjamini-Hochberg at fixed alpha.
4. Build the empirical-Bayes local false discovery rate from the z-score histogram (Efron) and compare which hypotheses it flags against Benjamini-Hochberg.
